## Clustering & Topic Analysis

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import numpy as np
from bertopic.representation import MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
import nltk
import pandas as pd
from transformers import AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
import geopandas as gpd
import geoplot as gplt
from shapely.geometry import Point
import contextily as ctx
from matplotlib.patches import Patch
import os

In [ ]:
os.makedirs("../data/plots", exist_ok=True)

### Load Embeddings & Texts

We start by loading the embeddings for our text samples and the respective samples themselves. We only use a subsample of 50.000, where Sascha's duplicates had been excluded.

In [ ]:
# load embeddings and text
embeddings = pd.read_csv("../data/encoded_unfalltext_first50000.csv")
embeddings = embeddings.to_numpy()
text = pd.read_csv("../data/D_Unfalltext_image_50000_subsample.csv")["Text"]
text = text.to_numpy()
# Download stopwords and extract german language. We need the stopwords for more refined representations of topics later.
nltk.download("stopwords")
stop_words = stopwords.words("german")

### Optional: Load json with parameter results

If you ran param_tuning.py before there will be a "param_results.json" in data containing different parameter configurations for bertopic. The topic models are evaluated regarding topic coherence, diversity, number of topics and outliers. In the following we will have a look at the results.

In [ ]:
with open('../data/param_results.json', 'r') as file:
    params = json.load(file)
params = pd.DataFrame(params)
params.info()
# Values need to be converted to numerics
params['coherence'] = pd.to_numeric(params['coherence'])
params['diversity'] = pd.to_numeric(params['diversity'])
params['n_topics'] = pd.to_numeric(params['n_topics'])
params['n_outliers'] = pd.to_numeric(params['n_outliers'])

In [ ]:
# Get a feeling for the numbers
params.describe()

In [ ]:
# configuration with fewest outlliers (has only two topics)
params.loc[params['n_outliers'].idxmin()]

In [ ]:
# configuration with highest coherence (has >20.000 outliers)
params.loc[params['coherence'].idxmax()]

In [ ]:
# configurations with at least 10 topics, sorted by coherence
params[params['n_topics'] == 2].sort_values("coherence")

### Load unfall_table to merge (it contains unfalltyp)

Later we will see how the generated topics correspond to certain accident types. For that we load the required csv-files containing each accident with their accident type label and the table containing the texts are loaded. They still have to be merged.

In [ ]:
unfall_table = pd.read_csv("../data/D_Unfall_RData.csv")
unfalltext_table = pd.read_csv("../data/D_Unfalltext_image_50000_subsample.csv")
# merge tables on UN_KEY
unfall_text_table_labels_merged = unfall_table.merge(unfalltext_table, on="UN_KEY", how="right")

In the following we will see two different topic model configurations. By running the param_tuning.py script you will get a good idea on how the hyperparameters of umap and hdbscan affect cluster size and amount. The first configuration develops fewer topics, while the second one has more topics, also capturing e.g. side mirror accidents and even some location specific accident topics. We will this focus more on the second model, which is referred to by a "_2" in the variable names.

Here we specify the above mentioned second and more granular topic model.

In [ ]:
# perform dimensionality reduction
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='manhattan', low_memory=False, force_approximation_algorithm=True, random_state=42)
# clustering
hdbscan_model = HDBSCAN(min_cluster_size=300, metric='manhattan', cluster_selection_method='eom', prediction_data=True)
# specify class based tf-idf approach for representing the topics
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = MaximalMarginalRelevance()
# include stopwords, we don't want them to show up in the topic representations
vectorizer_model = CountVectorizer(stop_words=stop_words)
# load our embedding model
embedding_model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True)

In [ ]:
# put parameters together into the model
topic_model = BERTopic(
    embedding_model=embedding_model,
    representation_model=representation_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto",
    ctfidf_model=ctfidf_model,
    vectorizer_model=vectorizer_model
    )

In [ ]:
# fit the topic model. We feed in our own precreated embeddings.
topics, probs = topic_model.fit_transform(text, embeddings)

In [ ]:
# get an overview
topic_model.get_topic_info()

Let's create some plots:

In [ ]:
# Label topics customly (mainly for the presentation, so better readable)
custom_labels = [
    "Parking 1",
    "Bicycle",
    "Crossroad, Crash",
    "Parking 2",
    "Parking 3",
    "Parking 4",
    "Truck",
    "Parking 5",
    "Bus",
    "Damaged city obj.",
    "Scooter",
    "Landsbergerstr.",
    "Schleißheimerstr.",
    "Intox.",
    "Fürstenriederstr.",
    "Dachauerstr.",
    "Parking 6",
    "Tram"
]

# Generate the heatmap
sim_heatmap = topic_model.visualize_heatmap()

# Remove bold title, adjust labeling and ticks, also feeding in custom labels from above
new_title = sim_heatmap.layout.title.text.replace("<b>", "").replace("</b>", "")
sim_heatmap.update_layout(
    title={'text': new_title, 'x': 0.5, 'xanchor': 'center'},
    xaxis=dict(
        tickmode="array", tickvals=list(range(18)), ticktext=custom_labels,
        tickangle=90
    ),
    yaxis=dict(
        tickmode="array", tickvals=list(range(18)), ticktext=custom_labels
    )
)

sim_heatmap.write_html("../data/plots/similarity_heatmap.html")

# for paper without title:
sim_heatmap.update_layout(
    title_text=None,  # Removes the title
    xaxis=dict(
        tickmode="array", tickvals=list(range(18)), ticktext=custom_labels,
        tickangle=90
    ),
    yaxis=dict(
        tickmode="array", tickvals=list(range(18)), ticktext=custom_labels
    )
)
sim_heatmap.write_image("../data/plots/similarity_heatmap_no_title.pdf")

In [ ]:
# reduce dimensions via umap the same way as we did for the topic model, only now we reduce to 2 dimensions to visualize
reduced_embeddings = UMAP(n_neighbors=35, n_components=2, min_dist=0.0, metric='manhattan', random_state=42).fit_transform(embeddings)
topic_map = topic_model.visualize_documents(text, reduced_embeddings=reduced_embeddings)
new_title = topic_map.layout.title.text.replace("<b>", "").replace("</b>", "")
topic_map.update_layout(title={'text': new_title, 'x': 0.5, 'xanchor': 'center'})

# again this is for the presentation so we remove the annotations since they don't reall matter
for trace in topic_map.data:
    if hasattr(trace, "hovertext") and trace.hovertext is not None:
        trace.hovertext = [""] * len(trace.hovertext)
    if hasattr(trace, "text") and trace.text is not None:
        trace.text = [""] * len(trace.text)
    if hasattr(trace, "name"):  
        trace.name = ""

topic_map.update_traces(showlegend=False)
topic_map.write_html("../data/plots/topic_map_no_ann.html")

# for report:
topic_map.update_layout(title_text=None)
for trace in topic_map.data:
    if hasattr(trace, "hovertext") and trace.hovertext is not None:
        trace.hovertext = [""] * len(trace.hovertext)
    if hasattr(trace, "text") and trace.text is not None:
        trace.text = [""] * len(trace.text)
    if hasattr(trace, "name"):  
        trace.name = ""

topic_map.update_traces(showlegend=False)
topic_map.write_image("../data/plots/topic_map_no_title.pdf")

In [ ]:
# for report
topic_map = topic_model.visualize_documents(text, reduced_embeddings=reduced_embeddings)
new_title = topic_map.layout.title.text.replace("<b>", "").replace("</b>", "")
topic_map.update_traces(showlegend=False)
topic_map.update_layout(title_text=None)
topic_map.write_image("../data/plots/topic_map_with_ann_no_title.pdf")

If we want to look at some samples from some topic we can do this by creating a dataframe and then picking some examples:

In [ ]:
df = pd.DataFrame({"Text": text, "Topic": topic_model.topics_})

In [ ]:
# e.g. look at examples from topic 15 (this topic is about Dachauerstraße)
df[df['Topic'] == 15].reset_index(drop=True)["Text"][0]

In [ ]:
# pick out representative texts
topic_model.get_representative_docs(15)

In [ ]:
# this shows the representative words and their occurence probability within the cluster
topic_model.get_topic(15)

In [ ]:
# we have some duplicate texts in the column Text which were not part of the subset we were supposed to exclude. So we left them in
# the topic model. Here we can check them and their labels.
pd.set_option("display.max_rows", None)
df[df["Text"].duplicated(keep=False)]

In [ ]:
# above we see that all duplicates have the same covariates, which means we can drop them - otherwise there will be a longer dataframe when merging one step later.
df = df.drop_duplicates(subset="Text", keep="first")

In [ ]:
unfall_text_table_labels_clusters_merged = unfall_text_table_labels_merged.merge(df, on="Text", how="right")

In [ ]:
pd.reset_option("display.max_rows")
unfall_text_table_labels_clusters_merged

We have some topics which appear to relate to locations. We have geo-information about each observation so let's overlay the points over the city of Munich!

In [ ]:
# we know from Sascha's R-File that US_GEO_X and US_GEO_Y have a certain fromat EPSG:4326
geometry = [Point(xy) for xy in zip(unfall_text_table_labels_clusters_merged['US_GEO_X'], unfall_text_table_labels_clusters_merged['US_GEO_Y'])]
# transform the subset of columns into GeoDataframe
gdf_points = gpd.GeoDataFrame(unfall_text_table_labels_clusters_merged.loc[:,("US_GEO_X", "US_GEO_Y", "Topic")], geometry=geometry, crs="EPSG:4326")
# remove empty points (around 400)
gdf_points = gdf_points[~gdf_points.is_empty]
# only filter points with topics associated with locations
filtered_points = gdf_points[gdf_points['Topic'].isin([11, 12, 14, 15])]
munich_gdf = gpd.GeoDataFrame(
    index=[0], 
    geometry=[Point(11.581981, 48.135124).buffer(0.01)],
    crs="EPSG:4326"
)
filtered_points = filtered_points.to_crs(epsg=3857)
munich_gdf = munich_gdf.to_crs(epsg=3857)
filtered_points['Topic'] = filtered_points['Topic'].astype('str')

First let's plot only the points in a coordinate system, with a map.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
gplt.pointplot(
    filtered_points,
    ax=ax,
    hue='Topic',
    legend=False,
    s=1.5
)
# figured those boundaries by just trying different numbers and seeing which show a good space
ax.set_xlim([1263168.278, 1309999.176])
ax.set_ylim([6112700.207, 6146060.611])
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
# reduce alpha so points are better visible
for im in ax.images:
    im.set_alpha(0.4)

plt.title("Location-dependent topics in Munich", fontsize=16)
# custom labels use only first two words of topic representations
custom_labels = {
    0: "Landsbergerstr.",
    1: "Schleißheimerstr., Lerchenauerstr.",
    2: "Fürstenriederstr., Würmtalstr.",
    3: "Dachauerstr.",
}
legend_patches = [
    Patch(color=color, label=label)
    for color, label in zip(plt.cm.viridis(np.linspace(0, 1, len(custom_labels))), custom_labels.values())
]
ax.legend(handles=legend_patches, title="Topics", fontsize=10, title_fontsize=12, loc="upper right")
plt.savefig("../data/plots/location_w_map.pdf", bbox_inches="tight", pad_inches=0)
plt.show()


In [ ]:
# for report: No title
fig, ax = plt.subplots(figsize=(10, 10))
gplt.pointplot(
    filtered_points,
    ax=ax,
    hue='Topic',
    legend=False,
    s=1.5
)
# figured those boundaries by just trying different numbers and seeing which show a good space
ax.set_xlim([1263168.278, 1309999.176])
ax.set_ylim([6112700.207, 6146060.611])
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
# reduce alpha so points are better visible
for im in ax.images:
    im.set_alpha(0.4)

plt.title(None)
# custom labels use only first two words of topic representations
custom_labels = {
    0: "Landsbergerstr.",
    1: "Schleißheimerstr., Lerchenauerstr.",
    2: "Fürstenriederstr., Würmtalstr.",
    3: "Dachauerstr.",
}
legend_patches = [
    Patch(color=color, label=label)
    for color, label in zip(plt.cm.viridis(np.linspace(0, 1, len(custom_labels))), custom_labels.values())
]
ax.legend(handles=legend_patches, title="Topics", fontsize=10, title_fontsize=12, loc="upper right")
plt.savefig("../data/plots/location_w_map_no_title.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

Now without map:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
gplt.pointplot(
    filtered_points,
    ax=ax,
    hue='Topic',
    legend=False,
    s=1.5
)
ax.set_xlim([1263168.278, 1309999.176])
ax.set_ylim([6112700.207, 6146060.611])
plt.title("Location-dependent topics in Munich", fontsize=16)
# custom labels use only first two words of topic representations
custom_labels = {
    0: "Landsbergerstr.",
    1: "Schleißheimerstr., Lerchenauerstr.",
    2: "Fürstenriederstr., Würmtalstr.",
    3: "Dachauerstr.",
}
legend_patches = [
    Patch(color=color, label=label)
    for color, label in zip(plt.cm.viridis(np.linspace(0, 1, len(custom_labels))), custom_labels.values())
]
ax.legend(handles=legend_patches, title="Topics", fontsize=10, title_fontsize=12, loc="upper right")
plt.savefig("../data/plots/location_no_map.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

## Similarity between human-labeled clusters and bertopic-generated clusters

We will now explore how the topics relate to our accident types. What we will do is group all texts used for clustering according to 1.What the topic model produced and 2. what the humans labeled the accidents as.
We then take both clusterings and encode them with the same embedding model we used for the topic modeling (jinna-embedding-v3). Then we compute cosine similarities and plot a matrix.

In [ ]:
# group by topic
grouped_by_human_label = unfall_text_table_labels_clusters_merged.groupby("Unfalltyp")["Text"].apply(list).to_dict()
# group by human prediction
grouped_by_bertopic_label = unfall_text_table_labels_clusters_merged.groupby("Topic")["Text"].apply(list).to_dict()

In [ ]:
# Helper function to create batches
def batchify(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

In [ ]:
# specify and load model as usual, and put it to gpu
MODEL_NAME = "jinaai/jina-embeddings-v3"
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

### Encode table grouped_by_human_label

Only run the following once - there will be no change in how the human-labeled clusters look.

In [ ]:
batch_size = 2048
encoded_human_labels = {}
for label, texts in tqdm(grouped_by_human_label.items(), desc="Processing labels"):
    all_embeddings = []
    for batch in batchify(texts, batch_size):
        embeddings = model.encode(batch)
        with torch.no_grad():
            # I now specidy task = "text-matching" because we are looking for similarity between texts
            embeddings = model.encode(batch, task="text-matching", max_length=2048)
        all_embeddings.extend(embeddings)
    
    encoded_human_labels[label] = all_embeddings

# apply mean pooling
mean_pooled_vectors_human_labels = {}
for label, vectors in encoded_human_labels.items():
    mean_vector = np.mean(vectors, axis=0)
    mean_pooled_vectors_human_labels[label] = mean_vector

In [ ]:
mean_pooled_vectors_human_labels = {str(label): vector for label, vector in mean_pooled_vectors_human_labels.items()}


In [ ]:
np.savez("../data/mean_pooled_vectors_human_labels.npz", **mean_pooled_vectors_human_labels)

### Encode table grouped_by_bertopic_label

The following needs to be executed everytime you evaluate a new topic model. Keep in mind that the tqdm time is overestimated in the beginning because it starts with the topics with more observations.

In [ ]:
batch_size = 2048
encoded_bertopic_labels = {}
for label, texts in tqdm(grouped_by_bertopic_label.items(), desc="Processing labels"):
    all_embeddings = []
    for batch in batchify(texts, batch_size):
        embeddings = model.encode(batch)
        with torch.no_grad():
            embeddings = model.encode(batch, task="text-matching", max_length=2048)
        all_embeddings.extend(embeddings)
    
    encoded_bertopic_labels[label] = all_embeddings

# apply mean pooling
mean_pooled_vectors_bertopic_labels = {}
for label, vectors in encoded_bertopic_labels.items():
    mean_vector = np.mean(vectors, axis=0)
    mean_pooled_vectors_bertopic_labels[label] = mean_vector

In [ ]:
mean_pooled_vectors_bertopic_labels = {str(label): vector for label, vector in mean_pooled_vectors_bertopic_labels.items()}

Don't forget to change the name of the file if you don't want to overwrite the embeddings of the old topic model clusters.

In [ ]:
np.savez("../data/mean_pooled_vectors_bertopic_labels.npz", **mean_pooled_vectors_bertopic_labels)

### Pairwise cosine Similarity

In [ ]:
loaded_data_humans = np.load('../data/mean_pooled_vectors_human_labels.npz')
mean_pooled_vectors_human_labels = {label: loaded_data_humans[label] for label in loaded_data_humans.files}
loaded_data_bertopic = np.load('../data/mean_pooled_vectors_bertopic_labels.npz')
mean_pooled_vectors_bertopic_labels = {label: loaded_data_bertopic[label] for label in loaded_data_bertopic.files}
# we remove outliers
del mean_pooled_vectors_bertopic_labels["-1"]

In [ ]:
vectors_bertopic = list(mean_pooled_vectors_bertopic_labels.values())
vectors_human = list(mean_pooled_vectors_human_labels.values())

labels_bertopic = list(mean_pooled_vectors_bertopic_labels.keys())
labels_human = list(mean_pooled_vectors_human_labels.keys())

cosine_sim_matrix = cosine_similarity(vectors_bertopic, vectors_human)

In [ ]:
# label the topics exactly as above for the other cosine similarity matrix
topic_labels = ["Parking 1", "Bicycle", "Crossroad, Crash", "Parking 2", "Parking 3", "Parking 4","Truck",
                "Parking 5", "Bus", "Damaged city obj.", "Scooter", "Landsbergerstr.", "Schleißheimerstr.",
                "Intox.", "Fürstenriederstr.", "Dachauerstr.", "Parking 6", "Tram"]

# our 7 accidents class indices
labels_human = ["A1", "A2", "A3", "A4", "A5", "A6", "A7"]

plt.figure(figsize=(5, 5))
sns.heatmap(
    cosine_sim_matrix,
    cmap='YlGnBu',
    xticklabels=labels_human,
    yticklabels=topic_labels,
    annot=False,
    fmt=".2f"
)
plt.title("Cosine Similarity between Human- and BERTopic-determined Classes", fontsize=11, pad=20)
plt.xlabel("Human-determined Classes")
plt.ylabel("BERTopic-determined Classes")

plt.savefig("../data/plots/heatmap_human_vs_bertopic_2_no_legend.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
# withoout title
topic_labels = ["Parking 1", "Bicycle", "Crossroad, Crash", "Parking 2", "Parking 3", "Parking 4","Truck",
                "Parking 5", "Bus", "Damaged city obj.", "Scooter", "Landsbergerstr.", "Schleißheimerstr.",
                "Intox.", "Fürstenriederstr.", "Dachauerstr.", "Parking 6", "Tram"]

# our 7 accidents class indices
labels_human = ["A1", "A2", "A3", "A4", "A5", "A6", "A7"]

plt.figure(figsize=(5, 5))
sns.heatmap(
    cosine_sim_matrix,
    cmap='YlGnBu',
    xticklabels=labels_human,
    yticklabels=topic_labels,
    annot=False,
    fmt=".2f"
)
plt.title(None)
plt.xlabel("Human-determined Classes")
plt.ylabel("BERTopic-determined Classes")


plt.savefig("../data/plots/heatmap_human_vs_bertopic_2_no_legend_no_title.pdf", bbox_inches="tight", pad_inches=0)
plt.show()